# ClassTail — locked paired calibration
Two CE warm-up rounds for tail arms; reference recipes active. No official test. No r20 before paired probe audit. Private inline W&B key persists in version history.

In [ ]:
import time
SESSION_T0=time.monotonic()

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


In [ ]:
MH=['v5_books.py', 'v3_methods.py', 'v4_mala.py', 'v7_nystrom.py', 'v9_anchor.py', 'v8_run.py', 'teacher_driver.py', 'v9_run.py', 'v5_run.py', 'new_methods.py', 'prepare.py', 'archs.py', 'v6_schema.py', '__init__.py', 'channel.py', 'v3_dense.py', 'v10_run.py', 'v6_run.py', 'dagsnet.py', 'metrics.py', 'v4_schema.py', 'new_fl.py', 'v7_run.py', 'model_packet.py', 'v6_geometry.py', 'v3_run.py', 'v8_hybrid.py', 'mobility.py', 'p2.py', 'pilot.py', 'v7_basis.py', 'v2_run.py', 'v3_math.py', 'v5_anchor.py', 'v4_run.py', 'v10_energy.py', 'v4_support.py', 'v2_methods.py', 'v2_math.py', 'teacher_base.py']
PROJ=['tail_math.py', 'tail_driver.py', 'tail_method.py', 'tail_verify.py']

import hashlib,json,shutil,time
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(2))
assert torch.__version__=='2.10.0+cu128'
hits=[p.parent for p in Path('/kaggle/input').rglob('tail_driver.py') if (p.parent/'train_0_X.npy').exists()]
assert len(hits)==1,hits
SRC=hits[0]; WORK=Path('/kaggle/temp/class-tail'); OUT=Path('/kaggle/working/runs')
SOURCE=Path('/kaggle/working/source'); SOURCE.mkdir(exist_ok=True)
for folder in ('mh','knowledge','papers/pilot/fixture','papers/class-tail/proj'):
    (WORK/folder).mkdir(parents=True,exist_ok=True)
manifest=json.loads((SRC/'source_manifest.json').read_text())
for name,sha in manifest.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,name
    shutil.copyfile(SRC/name,SOURCE/name)
for path in SRC.iterdir():
    if path.name in MH: dst=WORK/'mh'/path.name
    elif path.name in PROJ: dst=WORK/'papers/class-tail/proj'/path.name
    elif path.name=='meta.json': dst=WORK/'knowledge'/path.name
    else: dst=WORK/'papers/pilot/fixture'/path.name
    shutil.copyfile(path,dst)
fixture=json.loads((SRC/'manifest.json').read_text())
for name,sha in fixture['files'].items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,name
shutil.copyfile(SRC/'manifest.json',SOURCE/'fixture_manifest.json')
shutil.copyfile(SRC/'source_manifest.json',SOURCE/'source_manifest.json')
sys.path.insert(0,str(WORK));sys.path.insert(0,str(WORK/'papers/class-tail/proj'))
from mh.dagsnet import build_model,CFG
import numpy as np
for gpu in (0,1):
    torch.cuda.set_device(gpu)
    model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True).to(f'cuda:{gpu}').eval()
    x=torch.from_numpy(np.load(SRC/'train_0_X.npy',mmap_mode='r')[::100][:32].astype('float32')).to(f'cuda:{gpu}')
    z=model(x)[0];assert torch.isfinite(z).all()
    z.square().mean().backward();assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    del model,x,z;torch.cuda.empty_cache()
    print('T4 ClassTail forward/backward PASS',gpu,flush=True)

(WORK/"boundary_logging.py").write_text('"""Prevent preflight from advancing restored W&B steps; model source stays frozen."""\nimport runpy\nimport sys\nfrom pathlib import Path\n\n\ndef install(wandb):\n    run_class = wandb.sdk.wandb_run.Run\n    original = run_class.log\n\n    def log(self, data, *args, **kwargs):\n        if set(data) == {\'preflight/online\'} and not args and \'step\' not in kwargs:\n            kwargs = dict(kwargs, commit=False)\n        return original(self, data, *args, **kwargs)\n\n    run_class.log = log\n\n\nif __name__ == \'__main__\':\n    import wandb\n    install(wandb)\n    driver = sys.argv.pop(1)\n    sys.argv[0] = driver\n    sys.path.insert(0, str(Path(driver).parent))\n    runpy.run_path(driver, run_name=\'__main__\')\n')


In [ ]:
SEED=42
SESSION_HOURS=3.0
METHODS=['ct_local', 'ct_fixed', 'ct_own', 'ct_fl', 'ct_shuffle', 'ct_classmass', 'ct_gaussian', 'ct_dml']

OUT.mkdir(parents=True,exist_ok=True)
pending=list(METHODS);active={};summary={}
while active or pending:
    for gpu in (0,1):
        if gpu in active or not pending:continue
        remaining=SESSION_HOURS-(time.monotonic()-SESSION_T0)/3600
        assert remaining>.5,'calibration budget exhausted; inspect outputs before continuation'
        arm=pending.pop(0);folder=OUT/f'{arm}_s{SEED}';folder.mkdir(exist_ok=True)
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu),PYTHONPATH=str(WORK),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
        command=[sys.executable,str(WORK/'boundary_logging.py'),str(WORK/'papers/class-tail/proj/tail_driver.py'),
            '--arm',arm,'--seed',str(SEED),'--rounds','20','--eval_every','1',
            '--stop_after','2','--wandb_online','--max_hours',str(remaining-.25),'--out',str(OUT)]
        log=open(folder/'log.txt','w');proc=subprocess.Popen(command,cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT)
        active[gpu]=(arm,proc,log)
        print('start ClassTail calibration',arm,SEED,gpu,flush=True)
    for gpu,(arm,proc,log) in list(active.items()):
        if proc.poll() is None:continue
        log.close();del active[gpu]
        path=OUT/f'{arm}_s{SEED}/result.json'
        result=json.loads(path.read_text()) if path.exists() else {}
        rows=result.get('history',[])
        summary[arm]=dict(rc=proc.returncode,rounds=len(rows),target_rounds=20,
            mechanism_probe=rows[-1].get('diagnostics') if rows and arm=='ct_local' else None)
        assert proc.returncode==0 and len(rows)==2,('failed calibration',arm,summary[arm])
    time.sleep(2)
(OUT/'calibration_summary.json').write_text(json.dumps(summary,indent=2))
print('ClassTail calibration complete; paired mechanism gate must be audited before r20',summary,flush=True)
